# Refiner full-test ablations

This notebook evaluates the entire test split. Every batch is passed through the baseline and all refiner conditions before moving to the next batch.

- `correct`: matched visual contexts
- `zero`: all visual contexts are zeroed, but cross-attention remains active
- `mixed`: visual contexts are rotated between length-grouped samples


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import torch
from torchmetrics.text import WordErrorRate
from tqdm.auto import tqdm

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..")
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.datasets.vicocktail import load_vicocktail
from srcs.nets.backend.ctc import ctc_decode
from srcs.nets.e2e import get_model
from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform
from srcs.trainer.utils import (
    create_dataloader,
    load_config,
    move_batch,
    set_seed,
    update_wer,
)

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.yaml")
VSR_CHECKPOINT = os.path.join(
    PROJECT_ROOT, "checkpoints", "finetune_vsr_12", "final", "model.safetensors"
)
REFINER_CHECKPOINT = os.path.join(
    PROJECT_ROOT, "checkpoints", "refiner", "best.pt"
)
VISUAL_KEYS = ("visual_features", "h2_features", "h4_features")
CONDITIONS = ("correct", "zero", "mixed")
BATCH_SIZE = 8

config = load_config(CONFIG_PATH)
seed = config["training"]["seed"]
set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = config["evaluation"].get("amp", True) and device.type == "cuda"

test_dataset = load_vicocktail(
    test_fraction=1.0, splits=("test",), seed=seed
)["test"]
model_path, units_path = ensure_unigram()
text_transform = TextTransform(model_path, units_path)

evaluation_config = config["evaluation"].copy()
evaluation_config["batch_size"] = BATCH_SIZE
evaluation_config["seed"] = seed
test_loader = create_dataloader(
    test_dataset, text_transform, "test", evaluation_config, shuffle=True
)

base_model = get_model(
    "auto-vsr", text_transform.vocab_size, checkpoint=VSR_CHECKPOINT
).to(device).eval()
refiner = get_model(
    "refiner",
    text_transform.vocab_size,
    checkpoint=REFINER_CHECKPOINT,
    blank_id=text_transform.blank_id,
    ignore_id=text_transform.ignore_id,
).to(device).eval()

print(f"Device: {device}")
print(f"Test samples: {len(test_dataset)}")
print(f"Test batches: {len(test_loader)}")


In [ ]:
def align_to_reference(reference, hypothesis):
    rows = len(reference) + 1
    columns = len(hypothesis) + 1
    costs = [[0] * columns for _ in range(rows)]
    moves = [[None] * columns for _ in range(rows)]

    for row in range(1, rows):
        costs[row][0] = row
        moves[row][0] = "delete"
    for column in range(1, columns):
        costs[0][column] = column
        moves[0][column] = "insert"

    for row in range(1, rows):
        for column in range(1, columns):
            substitution_cost = int(reference[row - 1] != hypothesis[column - 1])
            candidates = (
                (costs[row - 1][column - 1] + substitution_cost, "match"),
                (costs[row - 1][column] + 1, "delete"),
                (costs[row][column - 1] + 1, "insert"),
            )
            costs[row][column], moves[row][column] = min(candidates, key=lambda item: item[0])

    aligned = [None] * len(reference)
    insertions = 0
    row, column = len(reference), len(hypothesis)

    while row > 0 or column > 0:
        move = moves[row][column]
        if move == "match":
            aligned[row - 1] = hypothesis[column - 1]
            row -= 1
            column -= 1
        elif move == "delete":
            row -= 1
        else:
            insertions += 1
            column -= 1

    return aligned, insertions


def align_batch(references, hypotheses):
    return [
        align_to_reference(reference, hypothesis)
        for reference, hypothesis in zip(references, hypotheses)
    ]


def new_transition_counts():
    return {"TP": 0, "TN": 0, "FP": 0, "FN": 0, "insertions": 0}


def update_transitions(counts, references, baseline_alignments, hypotheses):
    refined_alignments = align_batch(references, hypotheses)

    for reference, baseline_item, refined_item in zip(
        references, baseline_alignments, refined_alignments
    ):
        baseline_aligned, _ = baseline_item
        refined_aligned, refined_insertions = refined_item
        counts["insertions"] += refined_insertions

        for target, baseline_token, refined_token in zip(
            reference, baseline_aligned, refined_aligned
        ):
            baseline_correct = baseline_token == target
            refined_correct = refined_token == target

            if not baseline_correct and refined_correct:
                counts["TP"] += 1
            elif baseline_correct and refined_correct:
                counts["TN"] += 1
            elif baseline_correct and not refined_correct:
                counts["FP"] += 1
            else:
                counts["FN"] += 1


def transform_contexts(visual_contexts, condition, permutation):
    if condition == "zero":
        return {
            key: torch.zeros_like(value) if key in VISUAL_KEYS else value
            for key, value in visual_contexts.items()
        }

    if condition == "mixed":
        return {
            key: value.index_select(0, permutation) if key in VISUAL_KEYS else value
            for key, value in visual_contexts.items()
        }

    return visual_contexts


In [ ]:
wer_metrics = {name: WordErrorRate() for name in ("baseline", *CONDITIONS)}
loss_sums = {name: 0.0 for name in CONDITIONS}
transition_counts = {name: new_transition_counts() for name in CONDITIONS}
baseline_insertions = 0
sample_count = 0
mixed_length_difference_sum = 0
mixed_length_pair_count = 0
mixed_length_difference_max = 0
mixed_self_samples = 0

progress = tqdm(test_loader, desc="Full test")
for batch in progress:
    batch = move_batch(batch, device)
    batch_size = batch["videos"].size(0)

    with torch.inference_mode(), torch.autocast(
        device_type=device.type, enabled=amp_enabled
    ):
        baseline_logits, visual_contexts = base_model.get_contexts(
            batch["videos"], batch["video_lengths"]
        )

    input_lengths = visual_contexts["input_lengths"]
    baseline_outputs = {
        "logits": baseline_logits,
        "input_lengths": input_lengths,
    }
    update_wer(wer_metrics["baseline"], baseline_outputs, batch, text_transform)
    baseline_tokens = ctc_decode(
        baseline_logits, input_lengths, text_transform.blank_id
    )
    references = [
        label[: int(length)].detach().cpu().tolist()
        for label, length in zip(batch["labels"], batch["label_lengths"])
    ]
    baseline_alignments = align_batch(references, baseline_tokens)
    baseline_insertions += sum(item[1] for item in baseline_alignments)

    if batch_size > 1:
        permutation = torch.roll(
            torch.arange(batch_size, device=device), shifts=1
        )
    else:
        permutation = torch.arange(batch_size, device=device)
        mixed_self_samples += batch_size

    source_lengths = input_lengths.index_select(0, permutation)
    length_differences = (input_lengths - source_lengths).abs()
    mixed_length_difference_sum += int(length_differences.sum())
    mixed_length_pair_count += batch_size
    mixed_length_difference_max = max(
        mixed_length_difference_max, int(length_differences.max())
    )

    for condition in CONDITIONS:
        condition_contexts = transform_contexts(
            visual_contexts, condition, permutation
        )
        with torch.inference_mode(), torch.autocast(
            device_type=device.type, enabled=amp_enabled
        ):
            outputs = refiner(
                baseline_logits,
                condition_contexts,
                batch["labels"],
                batch["label_lengths"],
            )

        update_wer(wer_metrics[condition], outputs, batch, text_transform)
        refined_tokens = ctc_decode(
            outputs["logits"], outputs["input_lengths"], text_transform.blank_id
        )
        update_transitions(
            transition_counts[condition],
            references,
            baseline_alignments,
            refined_tokens,
        )
        loss_sums[condition] += outputs["loss"].item() * batch_size

    sample_count += batch_size
    progress.set_postfix(samples=sample_count)

results = {"baseline": {"wer": wer_metrics["baseline"].compute().item()}}
for condition in CONDITIONS:
    results[condition] = {
        "wer": wer_metrics[condition].compute().item(),
        "loss": loss_sums[condition] / sample_count,
    }

mean_length_difference = (
    mixed_length_difference_sum / mixed_length_pair_count
)
print(f"Evaluated samples: {sample_count}")
print(
    f"Mixed length difference: mean={mean_length_difference:.2f}, "
    f"max={mixed_length_difference_max}, self-paired={mixed_self_samples}"
)
print(f"Baseline WER: {results['baseline']['wer']:.6f}")
print(f"Baseline insertions: {baseline_insertions}")

for condition in CONDITIONS:
    counts = transition_counts[condition]
    baseline_wrong = counts["TP"] + counts["FN"]
    baseline_correct = counts["TN"] + counts["FP"]
    correction_rate = counts["TP"] / max(1, baseline_wrong)
    damage_rate = counts["FP"] / max(1, baseline_correct)
    print(
        f"{condition:>9} | WER={results[condition]['wer']:.6f} "
        f"loss={results[condition]['loss']:.6f} | "
        f"TP={counts['TP']} TN={counts['TN']} "
        f"FP={counts['FP']} FN={counts['FN']} | "
        f"correction={correction_rate:.4f} damage={damage_rate:.4f} | "
        f"insertions={counts['insertions']}"
    )


In [ ]:
display_names = {
    "baseline": "Baseline",
    "correct": "Correct",
    "zero": "Zero",
    "mixed": "Mixed",
}
names = ("baseline", *CONDITIONS)
wer_values = [100.0 * results[name]["wer"] for name in names]
colors = ["#777777", "#2E86AB", "#F6AE2D", "#D1495B"]

fig, axis = plt.subplots(figsize=(9, 4))
bars = axis.bar([display_names[name] for name in names], wer_values, color=colors)
axis.set_ylabel("WER (%)")
axis.set_title(f"Full test set ({sample_count} samples)")
axis.bar_label(bars, fmt="%.3f", padding=3)
axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(16, 4))
for axis, condition in zip(axes, CONDITIONS):
    counts = transition_counts[condition]
    matrix = torch.tensor(
        [[counts["FN"], counts["TP"]], [counts["FP"], counts["TN"]]],
        dtype=torch.float32,
    )
    total = matrix.sum().item()
    image = axis.imshow(matrix, cmap="Blues")

    for row in range(2):
        for column in range(2):
            count = int(matrix[row, column].item())
            percentage = 100.0 * count / max(1.0, total)
            axis.text(
                column, row, f"{count:,}\n{percentage:.1f}%",
                ha="center", va="center", color="black"
            )

    axis.set_xticks((0, 1), ("Wrong", "Correct"))
    axis.set_yticks((0, 1), ("Wrong", "Correct"))
    axis.set_xlabel("Refiner token")
    axis.set_ylabel("Baseline token")
    axis.set_title(display_names[condition])

fig.suptitle("Reference-aligned token transitions")
plt.tight_layout()
plt.show()

print("TP: baseline wrong -> refiner correct")
print("TN: baseline correct -> refiner correct")
print("FP: baseline correct -> refiner wrong")
print("FN: baseline wrong -> refiner wrong")
print("Insertions are reported separately and are not included in the 2x2 matrices.")
